# BIXI Hourly Feature Pipeline (Tasks 1-6)

## Objective
Build a reproducible, station-level hourly feature pipeline that prepares model-ready data for both inflow and outflow forecasting.

## Scope Completed in This Notebook
1. Load and validate silver rides data.
2. Build all-station inflow/outflow activity coverage.
3. Build and persist station-hour and station-day aggregates by `ride_year`.
4. Generate one graph-based station community assignment from the full rides dataset.
5. Build station + community hourly base features by `ride_year`.
6. Build one unified modeling dataset with both targets and candidate feature columns by `ride_year`.

## Output Datasets
- `data/features_temp/station_hourly_all`: station-hour inflow/outflow/net_flow aggregates, partitioned by `ride_year`.
- `data/features_temp/station_daily_all`: station-day inflow/outflow/net_flow aggregates, partitioned by `ride_year`.
- `data/features_temp/community_assignments`: one station-to-community mapping from all available rides.
- `data/features_temp/community_features/station_community_hourly_features`: joined station + community hourly base table, partitioned by `ride_year`.
- `data/features_temp/community_features/model_features_all`: unified modeling feature store with calendar, lag, and rolling features, partitioned by `ride_year`.

## How To Use The Processed Data Directly
- For base station/community signals, read `station_community_hourly_features`.
- For model training, read `model_features_all` and let the model builder choose feature subsets.
- Targets are already included in `model_features_all`: `station_inflow`, `station_outflow`, and `station_net_flow`.

## Execution Order
Run code cells sequentially from Task 1 through Task 6. Validation checks in each task fail fast when required inputs or outputs are missing.

In [1]:
from pathlib import Path
import numpy as np

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from graphframes import GraphFrame

# Stop any existing Spark session and restart cleanly.
try:
    SparkSession.getActiveSession().stop()
except Exception:
    pass

# Spark session for local processing (GraphFrames-compatible) with tuned memory.
spark = SparkSession.builder \
    .appName("bixi-timeseries-eda") \
    .master("local[*]") \
    .config("spark.driver.memory", "12g") \
    .config("spark.sql.shuffle.partitions", "96") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .config("spark.sql.shuffle.partitions.adaptive", "true") \
    .config("spark.sql.execution.arrow.maxRecordsPerBatch", "50000") \
    .config("spark.memory.offHeap.enabled", "false") \
    .config("spark.jars.packages", "io.graphframes:graphframes-spark4_2.13:0.10.0") \
    .getOrCreate()

spark.conf.set("spark.sql.session.timeZone", "America/Toronto")

BASE_TEMP_DIR = Path("data/features_temp")
BASE_TEMP_DIR.mkdir(parents=True, exist_ok=True)
COMMUNITY_FEATURE_DIR = BASE_TEMP_DIR / "community_features"
COMMUNITY_FEATURE_DIR.mkdir(parents=True, exist_ok=True)

print("Spark version:", spark.version)
print("spark.driver.memory:", spark.sparkContext.getConf().get("spark.driver.memory"))
print("spark.sql.shuffle.partitions:", spark.conf.get("spark.sql.shuffle.partitions"))

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/26 17:09:58 WARN Utils: Your hostname, YCLNVO-HOME, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/03/26 17:09:58 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/yuchen/project/bixi-analytics/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/yuchen/.ivy2.5.2/cache
The jars for the packages stored in: /home/yuchen/.ivy2.5.2/jars
io.graphframes#graphframes-spark4_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-dacbf390-34ec-4668-baad-d54baefee9ce;1.0
	confs: [default]
	found io.graphframes#graphframes-spark4_2.13;0.10.0 in central
	found io.graphframes#graphframes-graphx-spark4_2.13;0.10.0 in central
:: resolution report :: resolve 255ms :: artifacts dl 10ms
	

Spark version: 4.0.1
spark.driver.memory: 12g
spark.sql.shuffle.partitions: 96


In [2]:
rides_path = Path("data/silver/rides")
if not rides_path.exists():
    raise FileNotFoundError(f"Missing rides path: {rides_path}")

rides_df = spark.read.parquet(str(rides_path))

required_cols = [
    "start_canonical_station_id",
    "end_canonical_station_id",
    "start_time_ms",
    "end_time_ms",
]
missing_cols = [c for c in required_cols if c not in rides_df.columns]
if missing_cols:
    raise ValueError(f"rides_df missing required columns: {missing_cols}")

row_count = rides_df.count()
if row_count == 0:
    raise ValueError("rides_df is empty")

print("Rides partitions loaded from:", rides_path)
print("Row count:", f"{row_count:,}")
print("Column count:", len(rides_df.columns))
print("\nSchema:")
rides_df.printSchema()

candidate_station_cols = [c for c in rides_df.columns if "canonical_station_id" in c]
candidate_time_cols = [c for c in rides_df.columns if any(k in c.lower() for k in ["time", "date", "ts", "timestamp"])]
print("\nCandidate canonical station columns:", candidate_station_cols)
print("Candidate time/date columns:", candidate_time_cols)

rides_df.select(required_cols).limit(5).show(truncate=False)

Rides partitions loaded from: data/silver/rides
Row count: 27,588,048
Column count: 19

Schema:
root
 |-- end_station_key: string (nullable = true)
 |-- start_station_key: string (nullable = true)
 |-- start_station_name: string (nullable = true)
 |-- start_station_district: string (nullable = true)
 |-- start_station_latitude: double (nullable = true)
 |-- start_station_longitude: double (nullable = true)
 |-- end_station_name: string (nullable = true)
 |-- end_station_district: string (nullable = true)
 |-- end_station_latitude: double (nullable = true)
 |-- end_station_longitude: double (nullable = true)
 |-- start_time_ms: timestamp (nullable = true)
 |-- end_time_ms: timestamp (nullable = true)
 |-- start_station_name_norm: string (nullable = true)
 |-- end_station_name_norm: string (nullable = true)
 |-- start_coord_key: string (nullable = true)
 |-- end_coord_key: string (nullable = true)
 |-- start_canonical_station_id: string (nullable = true)
 |-- end_canonical_station_id: st

## Task 1 - Load and Inspect Silver Rides (All Stations)

### Objective
Confirm key columns needed for station-level time series modeling and build all-station directional activity coverage (instead of selecting one pilot station).

### Exit Criteria
- Identify canonical station id fields.
- Identify timestamp fields to build hourly and daily aggregates.
- Produce all-station inflow/outflow activity summary.

In [3]:
# Task 1: all-station directional coverage.
start_counts = (
    rides_df
    .where(F.col("start_canonical_station_id").isNotNull())
    .groupBy(F.col("start_canonical_station_id").alias("canonical_station_id"))
    .agg(F.count("*").alias("outflow_rides"))
)

end_counts = (
    rides_df
    .where(F.col("end_canonical_station_id").isNotNull())
    .groupBy(F.col("end_canonical_station_id").alias("canonical_station_id"))
    .agg(F.count("*").alias("inflow_rides"))
)

station_activity_sdf = (
    start_counts.join(end_counts, on="canonical_station_id", how="full")
    .fillna(0, subset=["outflow_rides", "inflow_rides"])
)

coverage_row = rides_df.select(
    F.count(F.when(F.col("start_canonical_station_id").isNotNull(), 1)).alias("start_id_non_null"),
    F.count(F.when(F.col("end_canonical_station_id").isNotNull(), 1)).alias("end_id_non_null"),
    F.count("*").alias("total_rows"),
).first()

if coverage_row["start_id_non_null"] == 0 or coverage_row["end_id_non_null"] == 0:
    raise ValueError("Canonical station coverage is zero for at least one direction")

summary_row = station_activity_sdf.select(
    F.count("canonical_station_id").alias("n_stations"),
    F.sum("outflow_rides").alias("sum_outflow_rides"),
    F.sum("inflow_rides").alias("sum_inflow_rides"),
).first()

print("All-station directional coverage:", dict(coverage_row.asDict()))
print("All-station summary:", dict(summary_row.asDict()))

print("Top 20 stations by total ride activity:")
station_activity_sdf \
    .withColumn("total_activity", F.col("outflow_rides") + F.col("inflow_rides")) \
    .orderBy(F.desc("total_activity")) \
    .limit(20) \
    .show(truncate=False)

All-station directional coverage: {'start_id_non_null': 27531543, 'end_id_non_null': 27410108, 'total_rows': 27588048}
All-station summary: {'n_stations': 1421, 'sum_outflow_rides': 27531543, 'sum_inflow_rides': 27410108}
Top 20 stations by total ride activity:


+--------------------+-------------+------------+--------------+
|canonical_station_id|outflow_rides|inflow_rides|total_activity|
+--------------------+-------------+------------+--------------+
|STN_0001            |249555       |224547      |474102        |
|STN_0002            |231039       |202846      |433885        |
|STN_0004            |181735       |147142      |328877        |
|STN_0005            |153618       |142298      |295916        |
|STN_0006            |138670       |137257      |275927        |
|STN_0007            |106231       |159161      |265392        |
|STN_0008            |128124       |134431      |262555        |
|STN_0003            |126015       |121818      |247833        |
|STN_0009            |123444       |120430      |243874        |
|STN_0010            |108241       |134179      |242420        |
|STN_0011            |94039        |141316      |235355        |
|STN_0012            |114013       |113727      |227740        |
|STN_0013            |119

## Task 2 - Capacity Resolution

> Status: Skipped in this run by request. We proceed directly to Task 3.

## Task 3 - Build Daily and Hourly Time Series (All Stations)

### Objective
Construct all-station inflow, outflow, and net-flow time series at hourly and daily levels using Spark-native transformations.

### Exit Criteria
- Create station-hour hourly table with inflow/outflow/net_flow.
- Create station-day daily table with inflow/outflow/net_flow.
- Report table sizes and preview top active station-hours without collecting full data to pandas.

In [4]:
# Task 3: stage hourly directional aggregates to disk by year.
outflow_stage_path = BASE_TEMP_DIR / "outflow_hourly_all"
inflow_stage_path = BASE_TEMP_DIR / "inflow_hourly_all"

ride_year_paths = sorted(rides_path.glob("ride_year=*"))
if not ride_year_paths:
    raise FileNotFoundError(f"No yearly partitions found under {rides_path}")

yearly_counts = []

for i, year_path in enumerate(ride_year_paths):
    ride_year = int(year_path.name.split("=")[1])
    rides_year_df = spark.read.parquet(str(year_path))

    outflow_hourly_year_sdf = (
        rides_year_df
        .where(F.col("start_canonical_station_id").isNotNull())
        .where(F.col("start_time_ms").isNotNull())
        .groupBy(
            F.col("start_canonical_station_id").alias("canonical_station_id"),
            F.date_trunc("hour", F.col("start_time_ms")).alias("ts_hour"),
        )
        .agg(F.count("*").alias("outflow"))
        .withColumn("ride_year", F.lit(ride_year).cast("int"))
    )

    inflow_hourly_year_sdf = (
        rides_year_df
        .where(F.col("end_canonical_station_id").isNotNull())
        .where(F.col("end_time_ms").isNotNull())
        .groupBy(
            F.col("end_canonical_station_id").alias("canonical_station_id"),
            F.date_trunc("hour", F.col("end_time_ms")).alias("ts_hour"),
        )
        .agg(F.count("*").alias("inflow"))
        .withColumn("ride_year", F.lit(ride_year).cast("int"))
    )

    write_mode = "overwrite" if i == 0 else "append"
    outflow_hourly_year_sdf.write.mode(write_mode).partitionBy("ride_year").parquet(str(outflow_stage_path))
    inflow_hourly_year_sdf.write.mode(write_mode).partitionBy("ride_year").parquet(str(inflow_stage_path))

    yearly_counts.append((ride_year, outflow_hourly_year_sdf.count(), inflow_hourly_year_sdf.count()))

if not yearly_counts:
    raise ValueError("No yearly data was staged")

print("Wrote staged hourly outflow to:", outflow_stage_path)
print("Wrote staged hourly inflow to:", inflow_stage_path)
print("Per-year staged row counts (ride_year, outflow_rows, inflow_rows):")
for rec in yearly_counts:
    print(rec)

Wrote staged hourly outflow to: data/features_temp/outflow_hourly_all
Wrote staged hourly inflow to: data/features_temp/inflow_hourly_all
Per-year staged row counts (ride_year, outflow_rows, inflow_rows):
(2024, 2957254, 3007757)
(2025, 3174055, 3224093)
(2026, 37402, 37770)


In [5]:
# Task 3: materialize final station-hour/day aggregates by year.
outflow_stage_path = BASE_TEMP_DIR / "outflow_hourly_all"
inflow_stage_path = BASE_TEMP_DIR / "inflow_hourly_all"
station_hourly_path = BASE_TEMP_DIR / "station_hourly_all"
station_daily_path = BASE_TEMP_DIR / "station_daily_all"

if not outflow_stage_path.exists() or not inflow_stage_path.exists():
    raise FileNotFoundError("Missing staged hourly outputs. Run Task 3 staging cell first.")

outflow_hourly_all_sdf = spark.read.parquet(str(outflow_stage_path))
inflow_hourly_all_sdf = spark.read.parquet(str(inflow_stage_path))

station_hourly_sdf = (
    outflow_hourly_all_sdf.join(
        inflow_hourly_all_sdf,
        on=["ride_year", "canonical_station_id", "ts_hour"],
        how="full",
    )
    .fillna(0, subset=["outflow", "inflow"])
    .withColumn("net_flow", F.col("inflow") - F.col("outflow"))
)

station_daily_sdf = (
    station_hourly_sdf
    .withColumn("ts_day", F.to_date("ts_hour"))
    .groupBy("ride_year", "canonical_station_id", "ts_day")
    .agg(
        F.sum("inflow").alias("inflow"),
        F.sum("outflow").alias("outflow"),
    )
    .withColumn("net_flow", F.col("inflow") - F.col("outflow"))
)

station_hourly_sdf.write.mode("overwrite").partitionBy("ride_year").parquet(str(station_hourly_path))
station_daily_sdf.write.mode("overwrite").partitionBy("ride_year").parquet(str(station_daily_path))

hourly_profile = station_hourly_sdf.select(
    F.count("*").alias("station_hour_rows"),
    F.countDistinct("canonical_station_id").alias("n_stations_hourly"),
    F.min("ts_hour").alias("min_ts_hour"),
    F.max("ts_hour").alias("max_ts_hour"),
).first()

daily_profile = station_daily_sdf.select(
    F.count("*").alias("station_day_rows"),
    F.countDistinct("canonical_station_id").alias("n_stations_daily"),
    F.min("ts_day").alias("min_ts_day"),
    F.max("ts_day").alias("max_ts_day"),
).first()

if hourly_profile["station_hour_rows"] == 0 or daily_profile["station_day_rows"] == 0:
    raise ValueError("Station aggregate outputs are unexpectedly empty")

print("Hourly table profile:", dict(hourly_profile.asDict()))
print("Daily table profile:", dict(daily_profile.asDict()))
print("Wrote station-hour features to:", station_hourly_path)
print("Wrote station-day features to:", station_daily_path)
station_hourly_sdf.groupBy("ride_year").count().orderBy("ride_year") .show(truncate=False)

Hourly table profile: {'station_hour_rows': 7483280, 'n_stations_hourly': 1421, 'min_ts_hour': datetime.datetime(2024, 1, 1, 13, 0), 'max_ts_hour': datetime.datetime(2026, 2, 7, 22, 0)}
Daily table profile: {'station_day_rows': 479873, 'n_stations_daily': 1421, 'min_ts_day': datetime.date(2024, 1, 1), 'max_ts_day': datetime.date(2026, 2, 7)}
Wrote station-hour features to: data/features_temp/station_hourly_all
Wrote station-day features to: data/features_temp/station_daily_all


+---------+-------+
|ride_year|count  |
+---------+-------+
|2024     |3565995|
|2025     |3860155|
|2026     |57130  |
+---------+-------+



## Task 4 - Community Statistics

### Objective
Generate community assignments and summary statistics first, then use them for feature engineering.

### Exit Criteria
- Build `communities_rec` from station flow graph.
- Report community distribution statistics (count, largest community size/share).
- Persist community assignments for reuse in the next step.

In [3]:
# Task 4: build one community assignment from full station-to-station flow.
rec_min_edge_weight = 40
rec_top_k_outgoing = 30
rec_lp_max_iter = 12

# Use all available rides to create a single stable community mapping.
rides_feature_base = spark.read.parquet(str(rides_path))

edges_base = (
    rides_feature_base.select(
        F.col("start_canonical_station_id").alias("src"),
        F.col("end_canonical_station_id").alias("dst")
    )
    .filter(F.col("src").isNotNull() & F.col("dst").isNotNull() & (F.col("src") != F.col("dst")))
)

edges_weighted = edges_base.groupBy("src", "dst").agg(F.count("*").alias("weight"))
edges_filtered = edges_weighted.filter(F.col("weight") >= F.lit(rec_min_edge_weight))

rank_w = Window.partitionBy("src").orderBy(F.col("weight").desc(), F.col("dst"))
edges_tuned = (
    edges_filtered
    .withColumn("rk", F.row_number().over(rank_w))
    .filter(F.col("rk") <= F.lit(rec_top_k_outgoing))
    .drop("rk")
)

vertices_tuned = (
    edges_tuned.select(F.col("src").alias("id"))
    .union(edges_tuned.select(F.col("dst").alias("id")))
    .distinct()
)

if edges_tuned.count() == 0 or vertices_tuned.count() == 0:
    raise ValueError("Community graph input is empty after tuning")

graph_rec = GraphFrame(vertices_tuned, edges_tuned)
communities_rec = graph_rec.labelPropagation(maxIter=rec_lp_max_iter)

community_stats_df = (
    communities_rec.groupBy("label")
    .agg(F.count("*").alias("station_count"))
    .orderBy(F.col("station_count").desc(), F.col("label"))
)

community_overview = community_stats_df.select(
    F.count("*").alias("n_communities"),
    F.sum("station_count").alias("n_stations_in_graph"),
    F.max("station_count").alias("largest_community_size"),
).first()

largest = float(community_overview["largest_community_size"] or 0)
total = float(community_overview["n_stations_in_graph"] or 0)
largest_share = (largest / total) if total else np.nan

community_assignments_path = BASE_TEMP_DIR / "community_assignments"
communities_rec.write.mode("overwrite").parquet(str(community_assignments_path))

print("Task 4 complete")
print(f"rec_min_edge_weight={rec_min_edge_weight}, rec_top_k_outgoing={rec_top_k_outgoing}, rec_lp_max_iter={rec_lp_max_iter}")
print("Saved community assignments:", community_assignments_path)
print("Community overview:", {**community_overview.asDict(), "largest_community_share": largest_share})

26/03/26 16:37:17 WARN LabelPropagation: Returned DataFrame is persistent and materialized!


Task 4 complete
rec_min_edge_weight=40, rec_top_k_outgoing=30, rec_lp_max_iter=12
Saved community assignments: data/features_temp/community_assignments
Community overview: {'n_communities': 35, 'n_stations_in_graph': 1297, 'largest_community_size': 211, 'largest_community_share': 0.16268311488049345}


## Task 5 - Build Station + Community Base Features

### Objective
Join station-hour flows to community assignments and create one base table with station and community hourly signals.

### Exit Criteria
- Build `station_community_hourly_features`.
- Persist base output under `data/features_temp`.
- Validate row counts and sample schema.

In [4]:
# Task 5: attach communities to station-hour flow and build base station+community table by year.
station_hourly_path = BASE_TEMP_DIR / "station_hourly_all"
community_assignments_path = BASE_TEMP_DIR / "community_assignments"

if not station_hourly_path.exists():
    raise FileNotFoundError("Missing station-hour table. Run Task 3 first.")
if not community_assignments_path.exists():
    raise FileNotFoundError("Missing community assignments. Run Task 4 first.")

station_hourly_base = spark.read.parquet(str(station_hourly_path))
communities_rec = spark.read.parquet(str(community_assignments_path))

station_hourly_with_community = (
    station_hourly_base
    .join(
        communities_rec.select(
            F.col("id").alias("canonical_station_id"),
            F.col("label").alias("community"),
        ),
        on=["canonical_station_id"],
        how="inner",
    )
)

station_hourly_features = station_hourly_with_community.select(
    "ride_year",
    "canonical_station_id",
    "community",
    "ts_hour",
    F.col("inflow").alias("station_inflow"),
    F.col("outflow").alias("station_outflow"),
    F.col("net_flow").alias("station_net_flow"),
)

community_hourly_features = (
    station_hourly_features
    .groupBy("ride_year", "community", "ts_hour")
    .agg(
        F.countDistinct("canonical_station_id").alias("community_active_stations"),
        F.sum("station_inflow").alias("community_inflow"),
        F.sum("station_outflow").alias("community_outflow"),
        F.sum("station_net_flow").alias("community_net_flow"),
    )
)

station_community_hourly_features = station_hourly_features.join(
    community_hourly_features,
    on=["ride_year", "community", "ts_hour"],
    how="left",
)

station_community_features_path = COMMUNITY_FEATURE_DIR / "station_community_hourly_features"
station_community_hourly_features.write.mode("overwrite").partitionBy("ride_year").parquet(str(station_community_features_path))

profile = station_community_hourly_features.select(
    F.count("*").alias("rows"),
    F.countDistinct("canonical_station_id").alias("n_stations"),
    F.countDistinct("community").alias("n_communities"),
    F.min("ts_hour").alias("min_ts_hour"),
    F.max("ts_hour").alias("max_ts_hour"),
).first()

if profile["rows"] == 0:
    raise ValueError("Task 5 output is empty")

print("Task 5 complete")
print("Saved:", station_community_features_path)
print("Profile:", dict(profile.asDict()))
station_community_hourly_features.groupBy("ride_year").count().orderBy("ride_year").show(truncate=False)

Task 5 complete
Saved: data/features_temp/community_features/station_community_hourly_features
Profile: {'rows': 7460963, 'n_stations': 1297, 'n_communities': 35, 'min_ts_hour': datetime.datetime(2024, 1, 1, 13, 0), 'max_ts_hour': datetime.datetime(2026, 2, 5, 3, 0)}
+---------+-------+
|ride_year|count  |
+---------+-------+
|2024     |3561532|
|2025     |3842550|
|2026     |56881  |
+---------+-------+



## Task 6 - Build Unified Modeling Dataset (Single Feature Store)

### Objective
Create one large modeling dataset containing both targets (`station_inflow`, `station_outflow`) and a broad set of candidate lag/rolling/calendar/community features, letting downstream model builders decide feature selection.

### Design Decision
- Keep a single dataset instead of separate inflow/outflow feature sets.
- Preserve candidate columns (including nulls from lag windows) to maximize downstream flexibility.

### Exit Criteria
- Build `model_features_all_sdf`.
- Persist to `data/features_temp/community_features/model_features_all`.
- Report profile and null-rate diagnostics for key feature groups.

In [2]:
# Task 6: build one unified modeling dataset by year (all candidate lag features + both targets).
base_features_path = COMMUNITY_FEATURE_DIR / "station_community_hourly_features"
if not base_features_path.exists():
    raise FileNotFoundError("Missing Task 5 output station_community_hourly_features. Run Task 5 first.")

model_features_all_path = COMMUNITY_FEATURE_DIR / "model_features_all"
base_year_paths = sorted(base_features_path.glob("ride_year=*"))
if not base_year_paths:
    raise FileNotFoundError(f"No yearly partitions found under {base_features_path}")

year_profiles = []

for i, year_path in enumerate(base_year_paths):
    ride_year = int(year_path.name.split("=")[1])
    base_sdf = spark.read.parquet(str(year_path)).withColumn("ride_year", F.lit(ride_year).cast("int"))

    # Keep only the two base windows for lag features.
    w_station = Window.partitionBy("canonical_station_id").orderBy("ts_hour")
    w_community = Window.partitionBy("community").orderBy("ts_hour")

    model_features_year_sdf = (
        base_sdf
        .withColumn("hour", F.hour("ts_hour"))
        .withColumn("dow", F.dayofweek("ts_hour") - F.lit(1))
        .withColumn("is_weekend", F.when(F.col("dow") >= 5, F.lit(1)).otherwise(F.lit(0)))
        .withColumn("hour_sin", F.sin(F.lit(2.0 * np.pi) * F.col("hour") / F.lit(24.0)))
        .withColumn("hour_cos", F.cos(F.lit(2.0 * np.pi) * F.col("hour") / F.lit(24.0)))
        .withColumn("dow_sin", F.sin(F.lit(2.0 * np.pi) * F.col("dow") / F.lit(7.0)))
        .withColumn("dow_cos", F.cos(F.lit(2.0 * np.pi) * F.col("dow") / F.lit(7.0)))
    )

    for l in [1, 2, 3, 4, 24, 48, 168]:
        model_features_year_sdf = model_features_year_sdf.withColumn(f"station_inflow_lag_{l}", F.lag("station_inflow", l).over(w_station))
        model_features_year_sdf = model_features_year_sdf.withColumn(f"station_outflow_lag_{l}", F.lag("station_outflow", l).over(w_station))
        model_features_year_sdf = model_features_year_sdf.withColumn(f"station_net_flow_lag_{l}", F.lag("station_net_flow", l).over(w_station))
        model_features_year_sdf = model_features_year_sdf.withColumn(f"community_inflow_lag_{l}", F.lag("community_inflow", l).over(w_community))
        model_features_year_sdf = model_features_year_sdf.withColumn(f"community_outflow_lag_{l}", F.lag("community_outflow", l).over(w_community))
        model_features_year_sdf = model_features_year_sdf.withColumn(f"community_net_flow_lag_{l}", F.lag("community_net_flow", l).over(w_community))

    write_mode = "overwrite" if i == 0 else "append"
    model_features_year_sdf.write.mode(write_mode).partitionBy("ride_year").parquet(str(model_features_all_path))

    year_profile = model_features_year_sdf.select(
        F.lit(ride_year).alias("ride_year"),
        F.count("*").alias("rows"),
        F.countDistinct("canonical_station_id").alias("n_stations"),
        F.countDistinct("community").alias("n_communities"),
        F.min("ts_hour").alias("min_ts_hour"),
        F.max("ts_hour").alias("max_ts_hour"),
        F.avg(F.col("station_inflow_lag_1").isNull().cast("double")).alias("null_rate_station_inflow_lag_1"),
        F.avg(F.col("station_inflow_lag_24").isNull().cast("double")).alias("null_rate_station_inflow_lag_24"),
        F.avg(F.col("station_net_flow_lag_168").isNull().cast("double")).alias("null_rate_station_net_flow_lag_168"),
        F.avg(F.col("community_inflow_lag_1").isNull().cast("double")).alias("null_rate_community_inflow_lag_1"),
        F.avg(F.col("community_net_flow_lag_168").isNull().cast("double")).alias("null_rate_community_net_flow_lag_168"),
    ).first()

    year_profiles.append(year_profile.asDict())

if not year_profiles:
    raise ValueError("Task 6 output is empty")

print("Task 6 complete")
print("Saved unified dataset:", model_features_all_path)
print("Per-year profile summary:")
for rec in year_profiles:
    print(rec)

26/03/26 17:10:19 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Task 6 complete
Saved unified dataset: data/features_temp/community_features/model_features_all
Per-year profile summary:
{'ride_year': 2024, 'rows': 3561532, 'n_stations': 1004, 'n_communities': 20, 'min_ts_hour': datetime.datetime(2024, 1, 1, 13, 0), 'max_ts_hour': datetime.datetime(2025, 1, 14, 5, 0), 'null_rate_station_inflow_lag_1': 0.0002819011593887125, 'null_rate_station_inflow_lag_24': 0.006763662378998701, 'null_rate_station_net_flow_lag_168': 0.04714179179072377, 'null_rate_community_inflow_lag_1': 5.615560943998257e-06, 'null_rate_community_net_flow_lag_168': 0.0009434142385917072}
{'ride_year': 2025, 'rows': 3842550, 'n_stations': 1152, 'n_communities': 34, 'min_ts_hour': datetime.datetime(2025, 1, 1, 0, 0), 'max_ts_hour': datetime.datetime(2026, 1, 29, 21, 0), 'null_rate_station_inflow_lag_1': 0.00029980091345590816, 'null_rate_station_inflow_lag_24': 0.007192098996759964, 'null_rate_station_net_flow_lag_168': 0.05017058984268259, 'null_rate_community_inflow_lag_1': 8.848